In [1]:
import pandas as pd
data = pd.read_excel("Poland_6digit_tree_plot_EN.xlsx")
data.head(3)

,isco08_1d,isco08_2d,isco08_3d,isco08_4d,isco08_6d,isco08_6d_task,potential25,task_color
0,1 - Managers,"11 - Chief executives, senior officials and le...",111 - Legislators and senior officials,1111 - Legislators,111101 - Member of Parliament,( 0.35 ) - Active participation in the works o...,Minimal Exposure,Low
1,1 - Managers,"11 - Chief executives, senior officials and le...",111 - Legislators and senior officials,1111 - Legislators,111101 - Member of Parliament,( 0.35 ) - Expressing one's position and submi...,Minimal Exposure,Low
2,1 - Managers,"11 - Chief executives, senior officials and le...",111 - Legislators and senior officials,1111 - Legislators,111101 - Member of Parliament,( 0.35 ) - Addressing the Presidium of the Sej...,Minimal Exposure,Low


In [3]:
import pandas as pd
import json

def build_hierarchy(df):
    hierarchy = {"name": "ISCO-08", "children": []}

    # Group up to occupation level
    grouped = df.groupby(["isco08_1d", "isco08_2d", "isco08_3d", "isco08_4d", "isco08_6d"])

    for (isco08_1d, isco08_2d, isco08_3d, isco08_4d, isco08_6d), group in grouped:
        # Level 1
        level1 = next((x for x in hierarchy["children"] if x["name"] == isco08_1d), None)
        if not level1:
            level1 = {"name": isco08_1d, "children": []}
            hierarchy["children"].append(level1)

        # Level 2
        level2 = next((x for x in level1["children"] if x["name"] == isco08_2d), None)
        if not level2:
            level2 = {"name": isco08_2d, "children": []}
            level1["children"].append(level2)

        # Level 3
        level3 = next((x for x in level2["children"] if x["name"] == isco08_3d), None)
        if not level3:
            level3 = {"name": isco08_3d, "children": []}
            level2["children"].append(level3)

        # Level 4 (no risk here)
        level4 = next((x for x in level3["children"] if x["name"] == isco08_4d), None)
        if not level4:
            level4 = {"name": isco08_4d, "children": []}
            level3["children"].append(level4)

        # Level 5 (isco08_6d → gets risk from potential25)
        risk_6d = group["potential25"].iloc[0]
        level5 = next((x for x in level4["children"] if x["name"] == isco08_6d), None)
        if not level5:
            level5 = {"name": isco08_6d, "risk": risk_6d, "children": []}
            level4["children"].append(level5)

        # Level 6 (isco08_6d_task → gets risk from task_color)
        for _, row in group.iterrows():
            level5["children"].append({
                "name": row["isco08_6d_task"],
                "risk": row["task_color"]
            })

    return hierarchy

# Build and export
hierarchy_data = build_hierarchy(data)
with open("output_data_EN.json", "w", encoding="utf-8") as f:
    json.dump(hierarchy_data, f, ensure_ascii=False, indent=2)

print("✅ JSON saved as output_data_EN.json")


✅ JSON saved as output_data_EN.json
